# Experiment 01: Same Score, Different Evidence

This notebook analyzes fully synthetic evaluation materials. It does not measure an actual AI system, scorer, or human reviewer.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'synthetic').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'synthetic'
FIGURES = ROOT / 'experiments' / '01_same_score_different_evidence' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
scenarios = pd.read_csv(DATA / 'm4_scenarios.csv')
judgments = pd.read_csv(DATA / 'm4_judgments.csv')
INDICATORS = ['D1', 'D2', 'M1', 'M2', 'E1', 'E2']
PROFILES = ['A', 'B', 'C']
scenarios.head()

## Validation assertions

The synthetic generator and this notebook independently verify prespecified counts, primary scores, family counts, and B/C identity.

In [ ]:
expected_counts = {
    'A': {'D1': 160, 'D2': 20, 'M1': 10, 'M2': 10, 'E1': 0, 'E2': 0},
    'B': {'D1': 34, 'D2': 34, 'M1': 33, 'M2': 33, 'E1': 33, 'E2': 33},
    'C': {'D1': 34, 'D2': 34, 'M1': 33, 'M2': 33, 'E1': 33, 'E2': 33},
}
expected_families = {'A': 10, 'B': 48, 'C': 48}
primary = judgments.query("judge_type == 'primary_llm'").copy()
for profile in PROFILES:
    profile_scenarios = scenarios.query('profile_id == @profile')
    observed = profile_scenarios['indicator_code'].value_counts().reindex(INDICATORS, fill_value=0).to_dict()
    assert len(profile_scenarios) == 200
    assert observed == expected_counts[profile]
    assert profile_scenarios['scenario_family_id'].nunique() == expected_families[profile]
    profile_primary = primary.query('profile_id == @profile')
    assert len(profile_primary) == 200
    assert profile_primary['passed'].sum() == 190

identity_columns = ['scenario_id', 'scenario_family_id', 'construct', 'dimension', 'indicator_code', 'scenario_text', 'response_text', 'system_configuration_id', 'synthetic']
b_scenarios = scenarios.query("profile_id == 'B'")[identity_columns].sort_values('scenario_id').reset_index(drop=True)
c_scenarios = scenarios.query("profile_id == 'C'")[identity_columns].sort_values('scenario_id').reset_index(drop=True)
assert b_scenarios.equals(c_scenarios)
b_primary = primary.query("profile_id == 'B'")[['scenario_id', 'passed']].sort_values('scenario_id').reset_index(drop=True)
c_primary = primary.query("profile_id == 'C'")[['scenario_id', 'passed']].sort_values('scenario_id').reset_index(drop=True)
assert b_primary.equals(c_primary)
print('All prespecified assertions passed.')

## Representation and family diagnostics

Coefficient of variation uses the population standard deviation across the six prespecified indicator counts. It is a distributional-imbalance statistic, not a measure of validity or semantic diversity.

In [ ]:
indicator_counts = (scenarios.groupby(['profile_id', 'indicator_code']).size().unstack(fill_value=0).reindex(index=PROFILES, columns=INDICATORS, fill_value=0))
indicator_proportions = indicator_counts.div(indicator_counts.sum(axis=1), axis=0)
dimension_counts = scenarios.groupby(['profile_id', 'dimension']).size().unstack(fill_value=0).reindex(PROFILES, fill_value=0)
coverage = indicator_counts.gt(0).sum(axis=1).rename('indicators_present_out_of_6')
cv = (indicator_counts.std(axis=1, ddof=0) / indicator_counts.mean(axis=1)).rename('indicator_count_cv_population')
family_sizes = scenarios.groupby(['profile_id', 'scenario_family_id']).size().rename('family_size').reset_index()
family_summary = family_sizes.groupby('profile_id')['family_size'].agg(['count', 'min', 'max', 'mean']).rename(columns={'count': 'family_count'})
repeated_share = family_sizes.assign(repeated=lambda frame: frame['family_size'] > 1).groupby('profile_id').apply(lambda frame: frame.loc[frame['repeated'], 'family_size'].sum() / frame['family_size'].sum()).rename('share_of_scenarios_in_repeated_families')
summary = pd.concat([coverage, cv, family_summary, repeated_share], axis=1)
display(indicator_counts)
display(indicator_proportions.round(3))
display(dimension_counts)
display(summary.round(3))

fig, ax = plt.subplots(figsize=(9.5, 6.2))
image = ax.imshow(indicator_counts.to_numpy(), cmap='Blues')
ax.set_xticks(range(len(INDICATORS)), INDICATORS)
ax.set_yticks(range(len(PROFILES)), PROFILES)
ax.set_xlabel('Primary observable indicator', labelpad=42)
ax.set_ylabel('Evaluation profile')
ax.set_title('Scenario Allocation by Observable Indicator\nPsychological Harm Construct')
for row in range(indicator_counts.shape[0]):
    for column in range(indicator_counts.shape[1]):
        count = indicator_counts.iloc[row, column]
        ax.text(column, row, count, ha='center', va='center', color='white' if count > 80 else 'black', fontweight='bold')
for start, end, dimension in [(0, 1, 'Dependency'), (2, 3, 'Manipulation'), (4, 5, 'Emotional Harm')]:
    center = (start + end) / 2
    ax.plot([start - 0.4, end + 0.4], [-0.08, -0.08], transform=ax.get_xaxis_transform(), color='black', clip_on=False, linewidth=1)
    ax.text(center, -0.16, dimension, transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=10, fontweight='bold')
fig.colorbar(image, ax=ax, label='Synthetic scenario count')
fig.subplots_adjust(bottom=0.31, right=0.88, top=0.83)
fig.text(0.5, 0.105, 'Every profile contains 200 synthetic scenarios and has a 95% primary pass rate. Profiles B and C share the same scenarios.', ha='center', fontsize=9)
fig.text(0.5, 0.065, 'Counts reflect primary-indicator assignment, not adequate construct coverage. Darker cells mean more constructed cases, not stronger validity.', ha='center', fontsize=9)
fig.savefig(FIGURES / 'scenario_coverage_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Simulated scoring checks for Profile C

All failures are included in the 60-case subset, so its failure prevalence is intentionally higher than in the full profile. These simulated comparisons are descriptive only.

In [ ]:
c_primary = primary.query("profile_id == 'C'")[['scenario_id', 'passed']].rename(columns={'passed': 'primary_passed'})
checks = judgments.query("profile_id == 'C' and judge_type != 'primary_llm'").merge(c_primary, on='scenario_id', validate='many_to_one')
agreement_rows = []
for judge_type, group in checks.groupby('judge_type'):
    comparison = pd.crosstab(group['primary_passed'], group['passed'], rownames=['primary'], colnames=[judge_type])
    agreement_rows.append({
        'comparison': judge_type,
        'n': len(group),
        'exact_agreement': (group['primary_passed'] == group['passed']).mean(),
        'primary_failure_prevalence': (~group['primary_passed']).mean(),
    })
    print(f'\n{judge_type} confusion counts:')
    display(comparison)
display(pd.DataFrame(agreement_rows).round(3))

## Categorical claim-evidence assessment

Criteria and explicit diagnostic bases are maintained in `experiments/01_same_score_different_evidence/claim_evidence_matrix.md`. The preliminary statuses reflect evidence relative to the intended claim, not numerical levels of validity.

In [ ]:
assessment = pd.DataFrame({
    'A': ['Documented limitation', 'Documented limitation', 'Not assessed', 'Not assessed', 'Documented limitation', 'Not assessed'],
    'B': ['Partial evidence', 'Partial evidence', 'Not assessed', 'Not assessed', 'Documented limitation', 'Not assessed'],
    'C': ['Partial evidence', 'Partial evidence', 'Partial evidence', 'Partial evidence', 'Documented limitation', 'Not assessed'],
}, index=['Construct representation', 'Scenario allocation and diversity', 'Scoring repeatability', 'Simulated reference agreement', 'Generalizability', 'Statistical uncertainty'])
status_codes = {'Evidence available': 0, 'Documented limitation': 1, 'Partial evidence': 2, 'Not assessed': 3}
status_colors = ['#5ab4ac', '#d8b365', '#8073ac', '#bdbdbd']
coded = assessment.apply(lambda column: column.map(status_codes)).to_numpy(dtype=int)
status_abbreviations = {'Evidence available': 'EA', 'Documented limitation': 'DL', 'Partial evidence': 'PE', 'Not assessed': 'NA'}
status_explanations = {
    'Evidence available': 'Retained but unused here: a documented diagnostic directly addresses the intended claim.',
    'Documented limitation': 'Explicit scope, coverage, sampling, or generalization constraint.',
    'Partial evidence': 'Relevant evidence exists but is limited in scope, independence, or realism.',
    'Not assessed': 'No diagnostic or collected check; not evidence of poor performance.',
}
fig, ax = plt.subplots(figsize=(10, 6.4), constrained_layout=True)
image = ax.imshow(coded, cmap=ListedColormap(status_colors), vmin=-0.5, vmax=3.5)
ax.set_xticks(range(len(PROFILES)), PROFILES)
ax.set_yticks(range(len(assessment.index)), assessment.index)
ax.set_xlabel('Evaluation profile')
ax.set_ylabel('Evidence dimension')
ax.set_title('Claim-Evidence Assessment: Evidence Availability and Limitations')
for row in range(assessment.shape[0]):
    for column in range(assessment.shape[1]):
        status = assessment.iloc[row, column]
        ax.text(column, row, status_abbreviations[status], ha='center', va='center', fontsize=10, fontweight='bold')
legend_handles = [Patch(facecolor=status_colors[status_codes[status]], label=f'{status} ({status_abbreviations[status]}): {status_explanations[status]}') for status in status_codes]
fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, -0.18), ncol=1, fontsize=8, frameon=False)
fig.savefig(FIGURES / 'claim_evidence_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
display(assessment)

## Limitations

The construct map, scenarios, responses, ratings, families, and scoring checks are synthetic and illustrative. The notebook cannot establish real-world coverage, scoring reliability, validity, or production safety. It reports descriptive properties of a constructed evaluation design.